In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [16]:
import pandas as pd 

import warnings
warnings.filterwarnings("ignore")

data = pd.read_csv('/content/drive/MyDrive/NoReC/combined_data.csv').head(10000)
data.head()

,id,label,txt_names,text,split
0,0,5,000000.txt,Rome S02\nToppen innen tv-drama akkurat nå! \n...,train
1,1,5,000001.txt,Twin Peaks - definitive gold box edition\nGull...,train
2,2,5,000002.txt,The Wire (sesong 1-4)\nThe Wire vil gjøre deg ...,train
3,3,4,000003.txt,"Mad Men (sesong 1)\nStilig, underholdende og s...",train
4,4,4,000004.txt,Mad Men (sesong 2)\nTV-underholdning av høyest...,train


In [17]:
X = list(data.text)
y = list(data.label)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

In [18]:
len(X_train),len(X_val), len(X_test)

(9000, 1000, 2000)

In [19]:
text_transformer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True, max_features=150000)

In [21]:
%%time
X_train_text = text_transformer.fit_transform(X_train)
X_test_text = text_transformer.transform(X_test)

CPU times: user 18.2 s, sys: 462 ms, total: 18.6 s
Wall time: 20.3 s


In [22]:
X_train_text.shape, X_test_text.shape

((9000, 150000), (2000, 150000))

In [24]:
logit = LogisticRegression(C=5e1, solver='lbfgs', multi_class='multinomial', random_state=17, n_jobs=4)

In [25]:
# Cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)

In [27]:
%%time
cv_results = cross_val_score(logit, X_train_text, y_train, cv=skf, scoring='f1_micro')

CPU times: user 1.08 s, sys: 350 ms, total: 1.43 s
Wall time: 2min


In [29]:
# It's nice to see that cross-validation is more or less stable across folds
print(cv_results)
print(cv_results.mean())

[0.54166667 0.535      0.51888889 0.52444444 0.52      ]
0.528


In [31]:
%%time
logit.fit(X_train_text, y_train)

CPU times: user 198 ms, sys: 208 ms, total: 406 ms
Wall time: 27.5 s


LogisticRegression(C=50.0, multi_class='multinomial', n_jobs=4, random_state=17)

In [32]:
test_preds = logit.predict(X_test_text)

In [33]:
from sklearn.metrics import accuracy_score
y_pred = test_preds
y_true = y_test
accuracy_score(y_true, y_pred)

0.7545

In [35]:
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.40      0.57        15
           1       0.93      0.56      0.70       143
           2       0.75      0.64      0.69       361
           3       0.72      0.80      0.76       710
           4       0.76      0.84      0.80       678
           5       0.87      0.56      0.68        93

    accuracy                           0.75      2000
   macro avg       0.84      0.63      0.70      2000
weighted avg       0.76      0.75      0.75      2000

